# How to Load External Data

<a href="https://colab.research.google.com/github/quprep/quprep/blob/main/examples/how-to/load_external_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/quprep/quprep/v0.10.0?labpath=examples%2Fhow-to%2Fload_external_data.ipynb)
<a href="https://account.qbraid.com?gitHubUrl=https://github.com/quprep/quprep/blob/main/examples/how-to/load_external_data.ipynb"><img src="https://qbraid-static.s3.amazonaws.com/logos/Launch_on_qBraid_white.png" height="23"></a>


QuPrep can load data from CSV files, NumPy arrays, OpenML, HuggingFace, and Kaggle — and stream large CSVs in chunks through a fitted pipeline.

**Optional dependencies:**
```bash
pip install quprep[huggingface]  # HuggingFaceIngester
pip install quprep[openml]       # OpenMLIngester
pip install quprep[kaggle]       # KaggleIngester
```

In [ ]:
!pip install -q quprep

In [ ]:
import tempfile
import warnings

import numpy as np
import pandas as pd

import quprep as qd
from quprep import QuPrepWarning

print(f"quprep {qd.__version__}")

## 1. CSV file

In [ ]:
df = pd.DataFrame({"a": [1.0, 2.0, 3.0], "b": [0.1, 0.2, 0.3], "c": [0.5, 0.6, 0.7]})
with tempfile.NamedTemporaryFile(suffix=".csv", mode="w", delete=False) as f:
    df.to_csv(f, index=False)
    csv_path = f.name

dataset = qd.CSVIngester().load(csv_path)
print(f"Shape : {dataset.data.shape}")

## 2. NumPy array

In [ ]:
rng = np.random.default_rng(0)
X = rng.uniform(0, 1, (50, 4))
y = (X[:, 0] > 0.5).astype(int)
dataset = qd.NumpyIngester().load(X, y=y)
print(f"Shape  : {dataset.data.shape}")
print(f"Labels : {np.bincount(y.astype(int))}")

## 3. Streaming CSV (chunked ingestion)

In [ ]:
big_df = pd.DataFrame({"x0": rng.uniform(0, 1, 200), "x1": rng.uniform(0, 1, 200)})
with tempfile.NamedTemporaryFile(suffix=".csv", mode="w", delete=False) as f:
    big_df.to_csv(f, index=False)
    big_csv = f.name

chunks = list(qd.CSVIngester().stream(big_csv, chunksize=50))
print(f"Total chunks : {len(chunks)}")
print(f"Chunk shapes : {[c.data.shape for c in chunks]}")

## 4. Pipeline.stream() — encode in chunks

In [ ]:
first_chunk = chunks[0]
with warnings.catch_warnings():
    warnings.simplefilter("ignore", QuPrepWarning)
    fitted_pipeline = qd.Pipeline(
        normalizer=qd.Scaler(strategy="minmax_pi"),
        encoder=qd.AngleEncoder(),
    ).fit(first_chunk)

encoded_chunks = list(fitted_pipeline.stream(big_csv, chunksize=50))
print(f"Encoded chunks : {len(encoded_chunks)}")
print(f"Circuits/chunk : {[len(c.encoded) for c in encoded_chunks]}")

## 5. OpenML (optional)

In [ ]:
try:
    from quprep.ingest.openml_ingester import OpenMLIngester
    ds = OpenMLIngester().load(61)  # Iris dataset ID on OpenML
    print(f"Shape  : {ds.data.shape}")
    print(f"Labels : {np.unique(ds.labels)}")
except ImportError:
    print("skipped — run: pip install quprep[openml]")
except Exception as e:
    print(f"skipped: {e}")